# 3. Build your own case

**This part of the workshop needs Navigate installed on your own machine.** If you have not set that up yet, work through [Setting up Navigate](https://zerocarbonshipping.github.io/navigate-zcs/tutorials/getting_started.html) first.

Some of the decks in this repository and part of this workshop were written with the help of an AI coding assistant. You can do the same: describe the case you want and let the assistant write a first-shot deck.

This notebook is about **prompting and running what comes back**. It does not build a
simulation itself (you have to do it).

Put whatever you generate under `simulations/user/`. That folder is gitignored,
so your experiments will not show up as changes to the repository.

```{admonition} Read this before you use any of it
:class: warning

**A prompt gets you a structure, not an answer.** What comes back will run, and
running is the easy part. Whether it means anything is on you. Each prompt below
was tested here by following it and running the deck it produced — the Arabian
Gulf one solves in about two seconds with no errors — but "it runs" is a statement
about the syntax, not about the shipping.

Before you interpret a result:

**Know where every input came from.** Open the `.inc` files the assistant wrote.
A value written as `Forecast("...")` is linked to the assumptions library and
moves with it. A bare number is not, and needs a comment saying where it came
from and why the library had none. An assistant will happily produce a deck full
of plausible numbers, and plausible numbers give plausible, wrong answers.

**Most of the library is a world average, not always your case.** The route is the
clearest example. All 23 operational profiles in
`assumptions/defaults/installation/Route/` are defined per *vessel class*, not
per geography: `operational_profile_tanker_127k_dwt` carries the speeds, time at
sea and laden/ballast split of the world's Suezmax fleet on average. **None of
them contains a distance at all**, and the only two ports they know are
`port_global` and `port_europe` — abstractions for regulatory scope, not places
on a map. So when your deck says "Ras Tanura to Ningbo", those names and that
distance are things *you* (or the AI assistant) supplied; the operating pattern underneath is still a
global average for the class. The same goes for default fuel prices (`fossil_fuel_oil_price_global`), fleet age distributions, orderbooks, and feedstock availability.

**So a case you build here is a structure to think with, not a study of a trade.**
```

## Three prompts

We propose here three example prompts that lead to a working Navigate deck.
Feel free to take some inspiration from them and create your own!

---

### 1. One trade, two ports

Real geography: a named route, a distance, a ship that bunkers at one end or the
other. This one has no regulation at all, so the fuel choice is made on cost
alone.

````text
Build me a Navigate deck under simulations/user/gulf_asia_tanker/.

THE CASE
- Fleet: Suezmax-class crude tankers, ~127,000 DWT, on a single trade
- Trade: Ras Tanura (Saudi Arabia) to Ningbo (China), laden out, ballast home
- Fuels: fuel oil and LNG as global commodities, plus e-ammonia and e-methanol
  produced in the Middle East on dedicated renewables
- Regulation: none. Neither port is in the EU. That is the point of the case:
  I want to see what the model does when nothing is pushing it.

The fleet starts on fuel oil. The model decides whether to retrofit to dual-fuel,
replace at end of life, or keep burning oil.

NUMBERS
Every number must come from assumptions/ and be referenced by name
(Forecast("..."), Curve("..."), Variable("...")), never copied in as a literal.
The library already covers this vessel class - use its own values:
    assumptions/defaults/installation/Fleet/tanker_127k_dwt.inc
    assumptions/defaults/installation/Route/operational_profile_tanker_127k_dwt.inc
Take structure from simulations/examples/example_4/ and none of its numbers.
Anything the library genuinely lacks - port-to-port distance, for instance -
mark INVENTED in a comment where it appears and list them in the .nav header.

DECK RULES
The .nav DEFINE and EVENTS blocks accept only Include and Load. Every node
declaration, Import and Copy goes in an .inc file that they include.

BEFORE YOU TELL ME IT WORKS
Run it with -d ./assumptions. Report the exit code and the log summary's warning
count. Tell me the fuel mix in the first year and the last, and say plainly
whether anything actually switched.
````

---

### 2. The world fleet, simplified

No geography here: the Center's reference scenarios work this way, with twenty-three
vessel classes on the library's own global operating profiles, and the only
"places" are two abstract ports that exist so the EU measures know what they
cover. This cuts that down to a size you can iterate on.

````text
Build me a Navigate deck under simulations/user/[YOUR FOLDER NAME]/.

THE CASE
A cut-down version of the Center's reference scenarios. No route, no map, no real
ports - I want the global picture, not a trade.

- Fleets: [TWO OR THREE CLASSES, e.g. tanker_127k_dwt and bulk_carrier_80k_dwt],
  imported whole from the library so each brings its own vessels, age
  distribution, orderbook, technologies and retrofit costs
- Geography: none of your own. Load DefaultVoyage for the per-class operating
  profiles and DefaultPort for the two abstract ports the library uses.
- Fuels and production: load the library's whole chain - DefaultEmission,
  DefaultFeedstock, DefaultFuel, DefaultProducer, DefaultPlant, DefaultRegion,
  DefaultSource, DefaultTransport
- Regulation: EU ETS and FuelEU at their library settings (DefaultRegulation)

NUMBERS
Everything comes from the library by construction here, because the deck is
almost entirely Load statements. If you find yourself writing a number, stop and
tell me why the library has none.

DECK RULES
- The .nav DEFINE and EVENTS blocks accept only Include and Load. Import Fleet
  statements go in an .inc file that DEFINE includes.
- Do NOT load DefaultConverterAvailable. It phases alternative vessel types in
  over time using wildcards like Fleet "tug*" that only resolve against the full
  twenty-three-fleet set, so it fails to parse on a cut-down deck. Say in the
  .nav header that every vessel type is therefore orderable from year one rather
  than phased in.

BEFORE YOU TELL ME IT WORKS
Run it with -d ./assumptions - expect a few minutes, not seconds. Report the exit
code and the warning count, and tell me the fuel mix at both ends of the horizon.
````

---

### 3. Two operators, one market

No new geography either: the same trade twice, run by two operators who make
different choices.

````text
Build me a Navigate deck under simulations/user/[YOUR FOLDER NAME]/.

THE CASE
Two operators in the same market, so I can see how much the operator matters
rather than the trade.

- Trade: [ORIGIN] to [DESTINATION], one route, both operators sailing it
- Operator A: [VESSEL CLASS], no energy-efficiency measures fitted
- Operator B: the same class and the same trade, but fitting the efficiency
  technologies the library offers for it
- Fuels: [WHICH ONES SHOULD COMPETE]
- Regulation: [EU ETS AND FUELEU / NONE]

Model them as two Fleet nodes sharing one Route, the way
simulations/examples/example_3/ does with its regular and efficient fleets. Same
costs, same route, same regulation - the only difference is what they fit.

NUMBERS
Every number must come from assumptions/ and be referenced by name
(Forecast("..."), Curve("..."), Variable("...")), never copied in as a literal.
Use the library's own Fleet file for the class, including its technology list and
retrofit costs. Take structure from the example deck and none of its numbers.
Mark anything INVENTED in a comment and list it in the .nav header.

DECK RULES
The .nav DEFINE and EVENTS blocks accept only Include and Load. Every node
declaration, Import and Copy goes in an .inc file that they include.

BEFORE YOU TELL ME IT WORKS
Run it with -d ./assumptions. Report the exit code and the warning count. Then
tell me what separates the two fleets by 2050 - fuel mix, emissions and the
freight rate - and whether the efficiency measures paid for themselves.
````

---

### Bonus: a prompt for writing prompts

If you are not sure what to ask for, hand this over first and let it interview
you.

````text
I want to build a simulation with Navigate, the shipping decarbonisation model in
this repository. Do not write any deck yet. Interview me first.

Read these so your questions are informed rather than generic:
  docs/reference_manual/overview.md          how a deck is put together
  simulations/examples/                      four worked cases of different shapes
  assumptions/defaults/installation/Fleet/   which vessel classes exist already

Then ask me, a few questions at a time: what question I actually want answered;
which fleet, and whether the library already covers that class; whether geography
matters for my question or whether a global run would do; which fuels should
compete; and which regulation applies, if any.

When you have enough, write out the prompt you would want to be given, show it to
me, and wait for me to approve it before building anything.
````

## Running what you get

In a terminal (and in the correct environment) run:

```bash
navigate simulations/user/<your_case>/<your_case>.nav -d ./assumptions
```

Two things worth knowing:

- **`-d ./assumptions` is needed** whenever the deck uses `Load`, `Import` or
  `Copy` — which any deck built on the library does. A fully self-contained deck
  like `quicktest` from [notebook 1](https://colab.research.google.com/github/zerocarbonshipping/navigate-zcs/blob/workshop-colab/docs/workshop/01-setup-and-quicktest.ipynb) needs no `-d` at all.
- **Output lands next to the deck**, not in your working directory. Navigate
  resolves the deck path first and writes everything relative to that.

### Where the plots are

`Load DefaultPlot` declares two `Plot` nodes, so a run fills two folders inside
the deck's own directory:

| where | what is in it |
|---|---|
| `<deck folder>/plots/` | the headline set, nine charts — among them fuel consumed, absolute emissions, installed power, producer development |
| `<deck folder>/plots_analysis/` | the fuller set, thirty-two charts — among them fleet evolution and conversions, speeds, emission intensity, expenses, bunker price and supply per port, plant production costs |

How many you get depends on what the deck contains. Two more files land beside
them: `<deck>.log`, and `plot_data.pkl`, which holds the solved model.

Open the PNGs from the file browser, or in a notebook with
`from IPython.display import Image; display(Image(filename=...))`. Which charts
appear is set by the `add_plot("...")` calls in the deck; the full list of labels
is in the appendix of the reference manual's
[Plot page](https://zerocarbonshipping.github.io/navigate-zcs/reference_manual/plot.html).

```{admonition} Old plots are not cleared
:class: note
Navigate overwrites a chart it regenerates but never deletes one it no longer
produces, so a PNG from an earlier configuration can sit in the folder looking
exactly as current as the rest. Delete the folder before a re-run if you have
changed which plots the deck asks for.
```

If the run finishes without errors, check the `Log summary` table at the foot of
`<deck>.log`: it counts warnings and errors, and is the quickest way to tell a
clean run from one that merely completed.

## Workshop notebooks

**Next: [4. Run the reference scenarios](https://colab.research.google.com/github/zerocarbonshipping/navigate-zcs/blob/workshop-colab/docs/workshop/04-run-a-reference-scenario.ipynb)**

Each link opens the notebook in Google Colab, in a fresh session, so run it
from the top.

1. [Test set-up](https://colab.research.google.com/github/zerocarbonshipping/navigate-zcs/blob/workshop-colab/docs/workshop/01-setup-and-quicktest.ipynb)
2. [Illustrating the Navigate logic with a simple case](https://colab.research.google.com/github/zerocarbonshipping/navigate-zcs/blob/workshop-colab/docs/workshop/02-vessel-case-study.ipynb)
3. **Build your own case** (this notebook)
4. [Run the reference scenarios](https://colab.research.google.com/github/zerocarbonshipping/navigate-zcs/blob/workshop-colab/docs/workshop/04-run-a-reference-scenario.ipynb)
5. [Change the numbers and see what moves](https://colab.research.google.com/github/zerocarbonshipping/navigate-zcs/blob/workshop-colab/docs/workshop/05-build-your-own-whatif.ipynb)